# 00 — Setup Verification

Notebook ini memverifikasi bahwa seluruh service Fase 2 sudah berjalan dan dapat diakses:

| Service    | URL                           | Cek |
|------------|-------------------------------|-----|
| ClickHouse | http://clickhouse:8123/ping   | ✅/❌ |
| MLflow     | http://mlflow:5000            | ✅/❌ |
| Jupyter    | (notebook ini sendiri)        | ✅   |

**Jalankan semua cell dari atas ke bawah** — jika tidak ada error, environment siap digunakan.

## 1. Cek Versi Library

In [ ]:
import sys, importlib

libs = [
    'clickhouse_connect', 'pandas', 'numpy', 'sklearn',
    'mlxtend', 'mlflow', 'matplotlib', 'seaborn', 'plotly'
]

print(f"Python  : {sys.version.split()[0]}")
for lib in libs:
    try:
        mod = importlib.import_module(lib)
        ver = getattr(mod, '__version__', 'n/a')
        print(f"  {lib:<22}: {ver}")
    except ImportError as e:
        print(f"  {lib:<22}: TIDAK DITEMUKAN — {e}")

## 2. Koneksi ke ClickHouse

In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/work/notebooks')

from utils import get_ch_client, read_sql, show_ch_tables

client = get_ch_client()
result = client.query('SELECT version() AS ch_version').result_rows
print(f"✅ ClickHouse tersambung — versi: {result[0][0]}")

In [ ]:
# Tampilkan tabel yang tersedia di tiap layer
for schema in ['bronze', 'silver', 'gold']:
    df = show_ch_tables(schema, client)
    print(f"\n── {schema.upper()} ({len(df)} tabel) ──")
    print(df.to_string(index=False))

In [ ]:
# Sample data dari gold layer
df_gold = read_sql('SELECT * FROM gold.gold_sales_daily LIMIT 5', client)
print(f"gold_sales_daily — {len(df_gold)} baris (sample)")
df_gold

## 3. Koneksi ke MLflow

In [ ]:
import mlflow, os

tracking_uri = os.getenv('MLFLOW_TRACKING_URI', 'http://mlflow:5000')
mlflow.set_tracking_uri(tracking_uri)

client_mf = mlflow.tracking.MlflowClient()
experiments = client_mf.search_experiments()

print(f"✅ MLflow tersambung — URI: {tracking_uri}")
print(f"   Jumlah experiment: {len(experiments)}")

if experiments:
    for exp in experiments:
        print(f"   - [{exp.experiment_id}] {exp.name}")

In [ ]:
# Test logging sederhana ke MLflow
with mlflow.start_run(run_name='setup_verification') as run:
    mlflow.set_tag('notebook', '00_setup_verification')
    mlflow.log_param('test_param', 'ok')
    mlflow.log_metric('test_metric', 1.0)
    run_id = run.info.run_id

print(f"✅ MLflow logging berhasil — run_id: {run_id}")
print(f"   Lihat di: {tracking_uri}")

## 4. Ringkasan

Jika semua cell di atas berjalan tanpa error, environment Fase 2 siap.

**Notebook selanjutnya:**

| Notebook | Topik | Model |
|----------|-------|-------|
| `01_clustering_customer_rfm.ipynb`     | Segmentasi pelanggan (RFM)   | K-Means |
| `02_classification_order_status.ipynb` | Prediksi status order        | Random Forest |
| `03_regression_revenue_forecast.ipynb` | Forecast pendapatan          | Linear/Ridge Regression |
| `04_association_market_basket.ipynb`   | Analisis keranjang belanja   | FP-Growth |